In [ ]:
#@title Copyright 2026 Google LLC
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Weather forecasting and storm risk nowcasting with `Earth Engine`, BigQuery ML, and Gemini

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/python-docs-samples/blob/feat/modernize-ppai-weather-forecasting/people-and-planet-ai/weather-forecasting/README.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fpython-docs-samples%2Ffeat%2Fmodernize-ppai-weather-forecasting%2Fpeople-and-planet-ai%2Fweather-forecasting%2FREADME.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/python-docs-samples/feat/modernize-ppai-weather-forecasting/people-and-planet-ai/weather-forecasting/README.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Agent Platform Workbench logo"><br> Open in Agent Platform Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/python-docs-samples/blob/feat/modernize-ppai-weather-forecasting/people-and-planet-ai/weather-forecasting/README.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<!-- Legacy Vertex AI redirect aliases: https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fpython-docs-samples%2Ffeat%2Fmodernize-ppai-weather-forecasting%2Fpeople-and-planet-ai%2Fweather-forecasting%2FREADME.ipynb and https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/python-docs-samples/feat/modernize-ppai-weather-forecasting/people-and-planet-ai/weather-forecasting/README.ipynb -->

<details>
<summary><b>Using Colab Enterprise or Agent Platform Workbench?</b> (Click for setup notes)</summary>

* **Agent Platform Workbench**: Clicking **Open in Agent Platform Workbench** opens the Workbench Instances list with a **Deploy** banner at the top. Make sure the Notebooks API (`notebooks.googleapis.com`) is enabled and you have at least one **Active** instance with internet access; check the box next to your running instance and click **Deploy** to import the notebook into JupyterLab.
* **Colab Enterprise VPC network**: If your organization enforces `constraints/compute.skipDefaultNetworkCreation`, create a VPC network and subnet with Private Google Access before connecting to a Colab Enterprise runtime:

```bash
gcloud compute networks create colab-network --subnet-mode=custom
gcloud compute networks subnets create colab-subnet --network=colab-network --region=us-central1 --range=10.0.0.0/24 --enable-private-ip-google-access
```

</details>

## Overview

Historically, weather forecasts relied on **physics-based Numerical Weather Prediction (NWP)**—solving millions of coupled differential equations to simulate atmospheric forces across a 3D grid. Much like calculating gravitational interactions across an increasing number of planets, physics simulations require massive supercomputers and hours of runtime; by the time calculations finish, local severe weather has often already shifted.

As introduced in the [**People and Planet AI: Weather Forecasting** video](https://www.youtube.com/watch?v=tfb6a51ff50), Google Research pioneered a **physics-free deep learning approach** with **MetNet**—training neural networks to recognize visual patterns of cloud and rain evolution directly from satellite and radar observations at `1 km` / `2-minute` resolution. Today, that research lineage powers zero-shot time-series foundation models in BigQuery ML (**TimesFM 2.5** via `AI.FORECAST`) and global probabilistic ensemble models (**Google DeepMind's WeatherNext 3**), letting you generate forecasts in seconds without training a custom CNN from scratch.

How do meteorologists and ML engineers validate a weather pipeline? Because extreme Category 4–5 hurricanes do not make landfall every day, you first test your model by **replaying a documented historical storm** (a **hindcast**, where you hold out 6 hours of known future rain as **ground-truth labels** to score accuracy), and then apply the same pipeline to **live, current weather happening right now**:

* **Part I (Steps 1–5) — Historical Hurricane Hindcast & Ground-Truth Evaluation (`Hurricane Melissa, October 27–28, 2025` or `Hurricane Helene, September 26–27, 2024`)**:
  * **Step 1 (Multi-Resolution Satellite & Terrain Alignment)**: Use Earth Engine's High Volume endpoint (`computePixels`) to align `2,000 m` **NOAA GOES** clouds, `10,000 m` **NASA GPM IMERG** rainfall (`mm/hr`), and **MERIT DEM** elevation onto a single grid, plus an alpha-blended **Rainbands-on-Terrain Overlay**.
  * **Step 2 (36-Hour Temporal Context & Intensity Buckets)**: Load the 36 hours of observed hourly rainfall leading up to landfall (`2025-10-27 06:00 UTC` to `2025-10-28 17:00 UTC`) into BigQuery using **Apache Beam**.
  * **Step 3 (6-Hour Zero-Shot Hindcast vs. Ground-Truth Labels)**: Use BigQuery ML `AI.FORECAST` (**TimesFM 2.5**) to hindcast the *subsequent 6 hours of the storm* (`2025-10-28 18:00–23:00 UTC`), apply non-negative **ReLU clamping (`max(0, y)`)**, and evaluate predictions against the actual observed 6-hour GPM ground-truth labels (**MAE**, **RMSE**, and **90% interval coverage**).
  * **Step 4 (Multimodal Flood-Risk Synthesis)**: Pass the 4-panel multi-band strip (`GOES + GPM + DEM + Overlay`) and the 6-hour forecast curve into **Gemini 3.8 Flash** to generate a structured flash-flood and mountain-runoff report.
  * **Step 5 (48-Hour 64-Member Storm Ensemble)**: Scale from a 6-hour point nowcast to a 48-hour, 64-member probabilistic storm ensemble using **Google DeepMind's WeatherNext 3**.
* **Part II (Step 6 — Finale) — Live Current Weather & Microclimates (`San Francisco, Palo Alto, San Jose, Santa Cruz & Monterey`)**:
  * **Step 6 (Live Bay Area Microclimates)**: Switch from historical hurricane replay to **live weather happening right now** across the San Francisco & Monterey Bay Area (`GOES-18 West`, 2.5 km `NOAA RTMA`, 72-hour `NOAA GFS`, and `MERIT DEM`), comparing 5 distinct microclimate zones or your own custom coordinates.

### Objectives

* Retrieve, align, and normalize multi-resolution Earth Engine satellite (`GOES-19`/`GOES-18`/`GOES-16`), precipitation (`GPM IMERG V07`), and elevation (`MERIT DEM`) rasters (`fetch_weather_patch.py`).
* Populate a BigQuery hourly storm time-series table using Apache Beam (`load_weather_time_series.py`).
* Execute zero-shot precipitation forecasting in BigQuery ML with `AI.FORECAST` (`TimesFM 2.5`) and evaluate holdout ground-truth accuracy (`forecast_precipitation.py`).
* Generate a structured `StormRiskAssessment` report from multi-band satellite imagery and tabular forecasts using `gemini-3.8-flash` (`analyze_storm_risk.py`).
* Compare 64-member **WeatherNext 3** storm ensembles and live 5-city **San Francisco & Monterey Bay** microclimate forecasts.

### Costs

This tutorial uses billable components of Google Cloud:

* [BigQuery](https://cloud.google.com/bigquery/pricing) (storage and `AI.FORECAST` query processing)
* [Dataflow](https://cloud.google.com/dataflow/pricing) (if running the Apache Beam pipeline on `DataflowRunner`; defaults to local `DirectRunner` in this notebook)
* [Vertex AI / Gemini API](https://cloud.google.com/vertex-ai/generative-ai/pricing) (`gemini-3.8-flash` multimodal token usage)
* [Cloud Storage](https://cloud.google.com/storage/pricing) (temporary staging bucket for BigQuery file loads)

## Get started

### Clone the sample repository and install dependencies

Fetch the `people-and-planet-ai/weather-forecasting` sample scripts and install the required Python packages into the current notebook kernel (~30 seconds).

In [ ]:
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/GoogleCloudPlatform/python-docs-samples.git"
BRANCH = "feat/modernize-ppai-weather-forecasting"
SAMPLE_DIR = "people-and-planet-ai/weather-forecasting"

if not os.path.exists("fetch_weather_patch.py"):
    if not os.path.exists("python-docs-samples"):
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none",
            "--sparse", "--branch", BRANCH, REPO_URL
        ], check=True)
        subprocess.run([
            "git", "-C", "python-docs-samples", "sparse-checkout", "set", SAMPLE_DIR
        ], check=True)
    os.chdir(os.path.join("python-docs-samples", SAMPLE_DIR))

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

Install the sample dependencies into the active Python environment using `%pip install`.

In [ ]:
%pip install --upgrade --quiet -r requirements.txt

### Authenticate your notebook environment

If you are running this notebook in Google Colab, authenticate with your Google Cloud user credentials. In Vertex AI Workbench or Colab Enterprise, Application Default Credentials (ADC) are preconfigured automatically.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Configure your Google Cloud project and historical storm preset

Set your `PROJECT_ID`, `BUCKET_NAME`, `BIGQUERY_DATASET`, and choose which historical hurricane landfall to replay in Steps 1–5: `melissa_2025` (Hurricane Melissa, October 28, 2025, Jamaica) or `helene_2024` (Hurricane Helene, September 27, 2024, Southern Appalachians).

> **Tip for Earth Engine access**: To pull live satellite pixels from Google Earth Engine, make sure your project is registered (free for research, education, and developer testing) at [console.cloud.google.com/earth-engine/configuration](https://console.cloud.google.com/earth-engine/configuration).

In [ ]:
import logging
import os
import subprocess
import warnings

logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning, module=r"pandas\.core\.arrays\.masked")
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning:pandas.core.arrays.masked"

PROJECT_ID = "example-project-id"  # @param {type:"string"}
LOCATION = "US"  # @param {type:"string"}
BUCKET_NAME = "example-weather-bucket"  # @param {type:"string"}
BIGQUERY_DATASET = "example_weather_forecasting"  # @param {type:"string"}
BIGQUERY_TABLE = "example_storm_observations"  # @param {type:"string"}
STORM_PRESET = "melissa_2025"  # @param ["melissa_2025", "helene_2024"]

if not PROJECT_ID or PROJECT_ID == "example-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or subprocess.check_output(
        ["gcloud", "config", "get-value", "project"], text=True
    ).strip()
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", LOCATION)
if not BUCKET_NAME or BUCKET_NAME == "example-weather-bucket":
    BUCKET_NAME = os.environ.get("BUCKET_NAME", f"{PROJECT_ID}-weather-forecasting")
BIGQUERY_DATASET = os.environ.get("BIGQUERY_DATASET", BIGQUERY_DATASET)
BIGQUERY_TABLE = os.environ.get("BIGQUERY_TABLE", BIGQUERY_TABLE)
STORM_PRESET = os.environ.get("STORM_PRESET", STORM_PRESET)

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
print(f"Project ID:       {PROJECT_ID}")
print(f"Staging Bucket:   gs://{BUCKET_NAME}")
print(f"BigQuery Dataset: {PROJECT_ID}.{BIGQUERY_DATASET}")
print(f"Storm Preset:     {STORM_PRESET}")

### Enable required Google Cloud APIs and create the staging bucket

Enable the BigQuery, Vertex AI (`aiplatform`), Earth Engine, and Dataflow APIs, and create a regional Cloud Storage bucket with soft-delete disabled for temporary BigQuery staging files (~30 seconds).

In [ ]:
!gcloud services enable bigquery.googleapis.com aiplatform.googleapis.com dataflow.googleapis.com earthengine.googleapis.com storage.googleapis.com --project={PROJECT_ID}

bucket_check = subprocess.run(
    ["gcloud", "storage", "buckets", "describe", f"gs://{BUCKET_NAME}", f"--project={PROJECT_ID}"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
if bucket_check.returncode != 0:
    !gcloud storage buckets create gs://{BUCKET_NAME} --project={PROJECT_ID} --location={LOCATION} --soft-delete-duration=0s
else:
    !gcloud storage buckets update gs://{BUCKET_NAME} --clear-soft-delete

## 1. Retrieve and visualize historical storm satellite, precipitation, and terrain patches

In **Part I (Steps 1–5)**, you replay a historical hurricane landfall—by default, **Hurricane Melissa at `2025-10-28T17:00:00Z`** as its Category 5 eyewall struck southwestern Jamaica (`18.1°N, 78.0°W`).

Studying the Earth at high resolution involves massive datasets with **different native pixel sizes and sensor units**. Instead of downloading terabytes of raw satellite archives, [`fetch_weather_patch.py`](fetch_weather_patch.py) uses Google Earth Engine's **High Volume endpoint** (`ee.data.computePixels`) to reproject, align, and normalize three public datasets onto a single grid (with an in-memory cache so re-running the cell is instantaneous):

1. **Panel 1 — Cloud & Moisture (`NOAA/GOES/19/MCMIPF` or `NOAA/GOES/16/MCMIPF`, `2,000 m` / `2 km` native resolution)**:
   * **What it is**: NOAA's Geostationary Operational Environmental Satellites orbit above the equator at the exact speed of Earth's rotation, scanning the Americas continuously (**GOES-19** replaced **GOES-16** as GOES-East in April 2025). Raw sensor digital numbers (`0–4,095`) are normalized into physical brightness temperatures (`Kelvin`) and `[0, 255]` RGB values.
   * **What you are looking at**: Bright white and cyan swirls show towering, ice-cold cloud tops and deep atmospheric water vapor spiraling into the eyewall.
2. **Panel 2 — Surface Precipitation Rate (`NASA/GPM_L3/IMERG_V07`, `~10,000 m` / `10 km` native resolution, in `mm/hr`)**:
   * **What it is**: NASA's Global Precipitation Measurement (**GPM**) constellation peers *through* the cloud tops using microwave and radar sensors to measure actual falling rain in millimeters per hour (`mm/hr`). In weather ML, GPM serves both as **historical input context** and as the **ground-truth output label** you want to predict.
   * **How to read the colors**: Dark navy is dry (`0 mm/hr`), cyan/green is light-to-moderate rain (`2–10 mm/hr`), yellow/amber is heavy rain (`15–25 mm/hr`), and red/magenta (`>30 mm/hr`) indicates torrential rainfall.
3. **Panel 3 — Terrain Elevation (`MERIT/DEM/v1_0_3`, resampled from `90 m` DEM, in meters)**:
   * **Why mountains matter (*orographic lift*)**: When moist hurricane winds slam into steep mountains—such as Jamaica's Blue Mountains (`2,256 m`) or North Carolina's Blue Ridge (`2,037 m`)—the terrain forces the air rapidly upward, cooling it and squeezing out intense rainfall on the windward slopes.
4. **Panel 4 — Rainbands Overlaid on Terrain (Alpha-Blended Spatial Composite)**:
   * **What it shows**: Alpha-blends the translucent GOES cloud deck and heavy GPM rainbands (`>= 1.2 mm/hr`) directly on top of the MERIT DEM mountain topography so you can immediately see where torrential rainbands collide with steep mountain ridges.

In [ ]:
import fetch_weather_patch
import visualize

patch_data = fetch_weather_patch.get_storm_patch_arrays(
    project_id=PROJECT_ID,
    storm_preset=STORM_PRESET,
    patch_size=64,
)
print(f"Storm:       {patch_data['storm_name']}")
print(f"Timestamp:   {patch_data['iso_time']}")
print(f"Satellite:   {patch_data['goes_satellite']}")
print(f"Data Source: {patch_data['source']}")

overview_fig = visualize.create_storm_overview_figure(patch_data)
overview_fig.show()

Package the 4 aligned panels (`[1. GOES | 2. GPM Rainfall | 3. MERIT DEM | 4. Rain-on-Terrain Overlay]`) into a single multi-band PNG file (`storm_multiband_strip.png`) for **Gemini 3.8 Flash** in Step 4. Passing the 3 unoccluded individual sensor panels alongside the 4th spatial overlay lets the multimodal model read exact rainfall and elevation colors without alpha-blending muddying the raw sensor channels.

In [ ]:
MULTIBAND_STRIP_PATH = "storm_multiband_strip.png"
png_bytes = visualize.export_multiband_strip_png_bytes(patch_data)
with open(MULTIBAND_STRIP_PATH, "wb") as f:
    f.write(png_bytes)
print(f"Saved 4-panel multi-band strip to {MULTIBAND_STRIP_PATH} ({len(png_bytes):,} bytes) for Gemini 3.8 Flash.")

## 2. Load 36 hours of historical storm observations into BigQuery with `Apache Beam`

A single satellite snapshot at `17:00 UTC` cannot tell a model how fast a storm is moving or whether rainfall is intensifying or decaying. As highlighted in the *People and Planet AI* video, a weather model needs **multiple historical time steps** (at least 3+ lag frames, such as `t-4h`, `t-2h`, and `t0`) to capture storm momentum. Here, you provide **36 hours of hourly storm observations leading up to landfall** (`2025-10-27 06:00 UTC` through `2025-10-28 17:00 UTC`).

### Why precipitation data requires intensity bucketing

Most random locations and hours on Earth have `0 mm/hr` of rain. If you train or evaluate a regression model on uniformly sampled weather data without accounting for this skew, a model that predicts `0 mm/hr` everywhere achieves >90% accuracy on clear days but fails completely during a hurricane. In custom neural pipelines (like the 31 stratified buckets from `0–38 mm/hr` in the video), engineers bucket rainfall by intensity (`Light < 2.5 mm/hr`, `Moderate 2.5–10 mm/hr`, `Heavy 10–30 mm/hr`, `Extreme > 30 mm/hr`) to ensure extreme events are represented.

Run [`load_weather_time_series.py`](load_weather_time_series.py) (`[START dataflow_weather_load_time_series]`) below to load the 36 hourly storm records into BigQuery via **Apache Beam** (~15–25 seconds on `DirectRunner`).

In [ ]:
# Expected duration: ~15-25 seconds on DirectRunner
!python load_weather_time_series.py \
  --project={PROJECT_ID} \
  --dataset={BIGQUERY_DATASET} \
  --table={BIGQUERY_TABLE} \
  --runner=DirectRunner \
  --temp_location=gs://{BUCKET_NAME}/temp

## 3. Hindcast the next 6 hours of the storm with BigQuery ML `AI.FORECAST` (`TimesFM 2.5`)

Given the **36 hours of historical storm inputs** up to `2025-10-28 17:00 UTC`, what does BigQuery ML predict for the **subsequent 6 hours of Hurricane Melissa (`2025-10-28 18:00 UTC` to `23:00 UTC`)**?

Instead of exporting 3 million `.npz` patches and training a custom 2D Conv/Deconv neural network from scratch, you use BigQuery ML's **zero-shot `AI.FORECAST`** function powered by Google Research's **TimesFM 2.5** foundation model.

* **Physical non-negativity (`ReLU` / `max(0, y)`)**: In the *People and Planet AI* video, the custom CNN used a **ReLU (`max(0, x)`)** output activation because **physical precipitation can never be negative (`< 0 mm/hr`)**. Even with a foundation model like `TimesFM 2.5`, wide 90% statistical prediction intervals can dip below zero as storm rain tapers off—so clamping lower bounds with `max(0.0, lower_bound_mm_hr)` enforces that same physical constraint.

In [ ]:
# Expected duration: ~10-20 seconds
!python forecast_precipitation.py {PROJECT_ID} {BIGQUERY_DATASET} {BIGQUERY_TABLE}

### Evaluate the 6-hour hindcast against observed ground-truth labels (`Inputs vs. Labels`)

Because this is a historical storm hindcast, you can check what **actually happened** during those 6 holdout hours (`2025-10-28 18:00–23:00 UTC`)! Just like splitting a dataset into **90% training inputs** and **10% unseen test labels**, the cell below compares `TimesFM 2.5`'s zero-shot forecast against the **actual observed GPM IMERG ground-truth rainfall** for `18:00–23:00 UTC`:

* **Blue solid line (`Observed GPM Inputs — 36h Past Context`)**: The 36 historical hours fed into `AI.FORECAST` (`Oct 27 06:00` to `Oct 28 17:00 UTC`).
* **Green solid line with diamonds (`Observed GPM Ground-Truth Labels — 6h Holdout`)**: The actual observed post-landfall hurricane rainfall (`Oct 28 18:00–23:00 UTC`) held back from the model.
* **Red dashed line & shaded band (`TimesFM 2.5 Hindcast & 90% Interval`)**: Shows how closely the zero-shot foundation model tracks the real post-landfall decay curve, along with **Mean Absolute Error (MAE)**, **Root Mean Squared Error (RMSE)**, and **90% confidence interval coverage**.

In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)
table_ref = f"{PROJECT_ID}.{BIGQUERY_DATASET}.{BIGQUERY_TABLE}"
hist_sql = f"SELECT observation_time, precipitation_mm_hr FROM `{table_ref}` ORDER BY observation_time ASC"
fc_sql = f"""
    SELECT forecast_timestamp, forecast_value AS forecasted_precipitation_mm_hr,
           prediction_interval_lower_bound AS lower_bound_mm_hr,
           prediction_interval_upper_bound AS upper_bound_mm_hr
    FROM AI.FORECAST(
      TABLE `{table_ref}`, data_col => 'precipitation_mm_hr',
      timestamp_col => 'observation_time', model => 'TimesFM 2.5',
      id_cols => ['storm_name'], horizon => 6, confidence_level => 0.90
    ) ORDER BY forecast_timestamp ASC
"""
historical_rows = [dict(r) for r in bq_client.query(hist_sql).result()]
forecast_rows = [dict(r) for r in bq_client.query(fc_sql).result()]
eval_metrics = visualize.evaluate_forecast_against_ground_truth(forecast_rows)
print(
    f"Holdout Evaluation (6h Unseen Ground Truth) -> MAE: {eval_metrics['mae_mm_hr']:.2f} mm/hr | "
    f"RMSE: {eval_metrics['rmse_mm_hr']:.2f} mm/hr | 90% Coverage: {eval_metrics['interval_coverage_pct']:.0f}%"
)
forecast_fig = visualize.create_forecast_comparison_figure(
    historical_rows, forecast_rows, "Hurricane Melissa (Oct 27-28, 2025)", eval_metrics["holdout_rows"]
)
forecast_fig.show()

## 4. Synthesize a multimodal hurricane flood-risk report with Gemini 3.8 Flash

Numbers in a database (`38.4 mm/hr`) only tell half the story—flood impact depends heavily on **geography**. Thirty millimeters of rain per hour over flat ocean is harmless, whereas the same rain falling onto already-saturated `2,000 m` mountain slopes funnels into narrow river gorges and triggers catastrophic flash floods and landslides.

In [`analyze_storm_risk.py`](analyze_storm_risk.py) (`[START googlegenaisdk_weather_analyze_storm_risk]`), you pass **both** the 4-panel hurricane multi-band strip (`storm_multiband_strip.png`) from Step 1 and the 6-hour `AI.FORECAST` trajectory from Step 3 to **Gemini 3.8 Flash** (`gemini-3.8-flash` on the `global` endpoint) using a Pydantic `StormRiskAssessment` schema:

* **`Display Name` (`flood_risk_level`)**: Overall flood severity classification for the landfall zone.
* **`Orographic Amplification Risk`**: Plain-language explanation of how the mountain ridges in Panel 3 & Panel 4 interact with the hurricane rainbands in Panel 2.
* **`Storm Summary` & bulleted actions**: Concrete, prioritized emergency response steps.

In [ ]:
os.environ.pop("WEATHER_FORECAST_CONTEXT", None)
!python analyze_storm_risk.py {PROJECT_ID} {MULTIBAND_STRIP_PATH}

## 5. Scale the hurricane forecast to a 48-hour, 64-member ensemble with `WeatherNext 3`

In Step 3, `AI.FORECAST` (`TimesFM 2.5`) gave you a fast 6-hour nowcast from a single station's rainfall history. What if you need to forecast an approaching hurricane **days in advance** across the entire globe?

Because small ripples in the atmosphere can shift a hurricane's track over 2 to 5 days, **Google DeepMind's WeatherNext 3** (`projects/gcp-public-data-weathernext/assets/weathernext_3_0_0_0p1deg`) runs **64 slightly different 3D atmospheric simulations ("ensemble members") every single hour**. In Google Earth Engine, all 64 simulations are pre-summarized into **percentile bands** across 114 variables:

* **`p50` (Ensemble Median — dark blue line)**: The middle scenario where half of the 64 simulations predict less rain and half predict more.
* **`p25` to `p75` (Interquartile Range — darker blue band)**: The middle 50% of all 64 simulations—the "most likely" corridor.
* **`p10` to `p90` (10th to 90th Percentile — outer light blue band & orange dotted line)**: Covers 80% of all 64 members. When the **`p90` tail-risk line** spikes much higher than `p50` (reaching `59.6 mm/hr` at `+24h` below), it warns forecasters days in advance that a subset of storm tracks produces extreme, life-threatening rainfall.

Run the cell below to query the live `weathernext_3_0_0_0p1deg` catalog in Earth Engine and render the 48-hour **WeatherNext 3 64-member ensemble plume chart** for the hurricane.

In [ ]:
wn_data = fetch_weather_patch.get_weathernext_ensemble_forecast(
    project_id=PROJECT_ID,
    region_name="Hurricane Melissa (Jamaica Landfall)",
    latitude=18.1,
    longitude=-78.0,
)
print(f"Collection ID:        {wn_data['collection_id']}")
print(f"Latest Live Asset ID: {wn_data['latest_asset_id']}")
print(f"Latest Init Time:     {wn_data['latest_init_utc']}")
print(f"Total Ensemble Bands: {wn_data['total_bands']}")
print(f"Precip Percentiles:   {', '.join(wn_data['precip_bands'])}")
for r in wn_data["ensemble_rows"]:
    print(
        f"  +{r['lead_hours']:02d}h ({r['valid_time_utc']}): "
        f"p50={r['p50_mm_hr']:4.1f} mm/hr | mean={r['mean_mm_hr']:4.1f} mm/hr | "
        f"p25-p75=[{r['p25_mm_hr']:4.1f}, {r['p75_mm_hr']:4.1f}] | "
        f"p10-p90=[{r['p10_mm_hr']:4.1f}, {r['p90_mm_hr']:4.1f}]"
    )

wn_fig = visualize.create_weathernext_ensemble_figure(wn_data)
wn_fig.show()

## 6. Live current weather and microclimate forecasting (`San Francisco, Palo Alto, San Jose, Santa Cruz & Monterey`)

Now that you have tested the pipeline on a historical hurricane in Part I, **Part II** switches to **live, current weather happening right now** across the **San Francisco and Monterey Bay Area**!

Northern and Central California are famous for dramatic **microclimates**, where cold Pacific water, gaps in the coastal hills, and `850 m` mountain ridges create completely different weather across five neighboring cities:

1. **San Francisco (`37.77°N, -122.42°W` — Golden Gate Marine Gap)**: Directly exposed to the cold Pacific Ocean through the Golden Gate gap; cool, breezy, and first to experience coastal fog (the marine layer).
2. **Palo Alto (`37.44°N, -122.14°W` — Mid-Peninsula Rain Shadow)**: Tucked behind the `750 m` Santa Cruz Mountains crest (`Skyline Blvd`), which blocks much of the Pacific fog and wrings out winter rain before it reaches the Bayside Peninsula.
3. **San Jose (`37.34°N, -121.89°W` — Inland Santa Clara Valley)**: Double-sheltered between the Santa Cruz Mountains to the west and the Diablo Range (`1,280 m`) to the east, making it warmer and significantly drier than the coast.
4. **Santa Cruz (`36.97°N, -122.03°W` — Windward Mountain Slopes)**: Faces south onto Monterey Bay with steep redwood mountains rising immediately behind town—where moist ocean breezes get pushed uphill (*orographic lift*). 
5. **Monterey (`36.60°N, -121.89°W` — Upwelling Bay Peninsula)**: Surrounded by the deep, cold waters of the Monterey Submarine Canyon, keeping temperatures cool and humidity high year-round.

Run the cell below to pull the live **GOES-18 West** satellite scan (`NOAA/GOES/18/MCMIPF`), **2.5 km NOAA RTMA** mesoscale observations (`NOAA/NWS/RTMA`), **72-hour NOAA GFS** forecasts (`NOAA/GFS0P25`), and **MERIT DEM** terrain across all five cities (or customize `LOCAL_REGION_NAME`, `LOCAL_LATITUDE`, and `LOCAL_LONGITUDE` for your own city).

In [ ]:
LOCAL_REGION_NAME = "San Francisco & Monterey Bay Area, CA"  # @param {type:"string"}
LOCAL_LATITUDE = 37.7749  # @param {type:"number"}
LOCAL_LONGITUDE = -122.4194  # @param {type:"number"}
LOCAL_REGION_NAME = os.environ.get("LOCAL_REGION_NAME", LOCAL_REGION_NAME)

local_result = fetch_weather_patch.get_local_weather_forecast(
    project_id=PROJECT_ID,
    region_name=LOCAL_REGION_NAME,
    latitude=float(LOCAL_LATITUDE),
    longitude=float(LOCAL_LONGITUDE),
)
local_patch = local_result["patch_data"]
print(f"Region:      {local_patch['storm_name']}")
print(f"Data Source: {local_patch['source']}\n")
print("5-City Bay Area & Monterey Bay Microclimate Comparison (Live 2.5km RTMA + 24h GFS):")
for m in local_result["microclimate_rows"]:
    print(
        f"  {m['city']:14s} ({m['microclimate_zone']:25s}) | Elev {m['elevation_m']:4.0f}m | "
        f"Now: {m['current_temp_c']:4.1f}°C ({m['current_temp_f']:4.1f}°F), wind {m['current_wind_ms']:3.1f} m/s | "
        f"+24h: {m['forecast_24h_temp_c']:4.1f}°C ({m['forecast_24h_temp_f']:4.1f}°F), RH {m['forecast_24h_rh_pct']:2.0f}%"
    )

micro_fig = visualize.create_bay_area_microclimate_figure(local_result["microclimate_rows"])
micro_fig.show()
local_fig = visualize.create_storm_overview_figure(local_patch)
local_fig.show()

### Synthesize a live `Bay Area` microclimate and weather report with Gemini 3.8 Flash

Package the 4-panel Bay Area & Monterey corridor multi-band strip (`local_weather_multiband_strip.png`) and pass it alongside the 5-city microclimate comparison and 72-hour `NOAA/GFS0P25` forecast trajectory into [`analyze_storm_risk.py`](analyze_storm_risk.py). Notice how `gemini-3.8-flash` contrasts the cool, humid marine gaps (`San Francisco`, `Monterey`) and coastal slopes (`Santa Cruz`) against the warmer, rain-shadowed valleys (`Palo Alto`, `San Jose`).

In [ ]:
LOCAL_STRIP_PATH = "local_weather_multiband_strip.png"
local_png_bytes = visualize.export_multiband_strip_png_bytes(local_patch)
with open(LOCAL_STRIP_PATH, "wb") as f:
    f.write(local_png_bytes)
print(f"Saved 4-panel multi-band strip to {LOCAL_STRIP_PATH} ({len(local_png_bytes):,} bytes) for Gemini 3.8 Flash.")

os.environ["WEATHER_FORECAST_CONTEXT"] = local_result["forecast_context"]
!python analyze_storm_risk.py {PROJECT_ID} {LOCAL_STRIP_PATH}

## Cleaning up

To avoid incurring ongoing charges to your Google Cloud account for the resources created in this tutorial, delete the BigQuery dataset and temporary Cloud Storage staging files.

In [ ]:
!bq rm -r -f -d {PROJECT_ID}:{BIGQUERY_DATASET}
!gcloud storage rm --recursive gs://{BUCKET_NAME}/temp